# Ride Demand Forecasting Data Prep Engine

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sqlite3 
import requests
import os
from ydata_profiling import ProfileReport
from sklearn.impute import SimpleImputer , MissingIndicator , KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder , OneHotEncoder , LabelEncoder 
from sklearn.preprocessing import Binarizer
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import PowerTransformer
from scipy.stats import skew
from sklearn.compose import ColumnTransformer

c:\PJ Things\AI ML and Data Science\Data Pre-processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\bhatt\AppData\Local\Temp\ipykernel_8888\895917257.py:9: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


## 1. Data understanding & loading

- Load CSV,JSON and SQL files

CSV

In [2]:
df = pd.read_csv('riders.csv')
C_Data = pd.DataFrame(df)
C_Data

,rider_id,name,age,gender,city,signup_date,total_rides,cancelled_rides,avg_rating
0,R0001,Aarav Das,23,Male,Pune,2020-06-29,56,0,3.76
1,R0002,Ishaan Nair,39,Female,Mumbai,2019-11-23,70,5,4.12
2,R0003,Kavya Reddy,34,Male,Pune,2023-05-04,45,9,3.76
3,R0004,Aarav Nair,19,Other,Kolkata,2019-07-28,464,5,3.19
4,R0005,Diya Reddy,27,Male,Ahmedabad,2021-05-31,294,30,3.53
...,...,...,...,...,...,...,...,...,...
295,R0296,Reyansh Patel,37,Other,Hyderabad,2021-02-20,390,58,3.97
296,R0297,Saanvi Gupta,47,Male,Pune,2023-09-20,396,46,3.87
297,R0298,Vivaan Singh,35,Female,Ahmedabad,2020-03-25,65,11,3.88
298,R0299,Ishaan Gupta,33,Other,Kolkata,2020-09-04,457,0,3.92


JSON

In [3]:
with open("trips.json") as file:
    raw = json.load(file)

J_Data = pd.DataFrame(raw)
J_Data

,trip_id,rider_id,zone,distance_km,duration_min,fare_amount,payment_mode,ride_date,surge_flag
0,T00001,R0037,Zone_10,11.83,74.59,104.88,Cash,2023-11-13,0
1,T00002,R0104,Zone_9,3.86,35.59,40.48,Cash,2023-07-28,1
2,T00003,R0045,Zone_8,4.70,31.03,46.39,Cash,2024-01-14,1
3,T00004,R0089,Zone_2,11.06,59.48,257.64,Cash,2023-12-13,0
4,T00005,R0003,Zone_5,7.28,67.59,72.74,UPI,2023-03-15,1
...,...,...,...,...,...,...,...,...,...
1995,T01996,R0171,Zone_6,0.91,6.44,22.12,UPI,2023-03-20,0
1996,T01997,R0287,Zone_5,11.40,103.03,193.75,Cash,2024-06-25,1
1997,T01998,R0254,Zone_9,4.94,39.32,58.45,Cash,2024-03-23,0
1998,T01999,R0267,Zone_2,7.76,51.43,74.92,UPI,2023-10-29,1


SQL

In [4]:
conn = sqlite3.connect('city_zones.db')  
with open('city_zones.sql', 'r') as f:
    sql_script = f.read()

conn.executescript(sql_script)
conn.commit()

In [5]:
df2 = pd.read_sql_query("SELECT * FROM city_zones", conn)
S_Data = pd.DataFrame(df2)
S_Data

,zone_name,population_density,traffic_index,avg_speed_kmph,zone_type
0,Zone_1,4921,2.43,30.9,Residential
1,Zone_2,6371,0.91,58.4,Residential
2,Zone_3,12971,2.11,38.0,Business
3,Zone_4,4038,2.46,48.2,Business
4,Zone_5,2590,1.31,43.9,Business
5,Zone_6,14627,0.54,37.6,Mixed
6,Zone_7,11070,0.64,45.2,Industrial
7,Zone_8,4516,1.93,31.0,Business
8,Zone_9,11037,1.83,56.1,Mixed
9,Zone_10,2440,1.54,35.4,Industrial


- Display

First 5 rows for each dataset

In [6]:
C_Data.head()

,rider_id,name,age,gender,city,signup_date,total_rides,cancelled_rides,avg_rating
0,R0001,Aarav Das,23,Male,Pune,2020-06-29,56,0,3.76
1,R0002,Ishaan Nair,39,Female,Mumbai,2019-11-23,70,5,4.12
2,R0003,Kavya Reddy,34,Male,Pune,2023-05-04,45,9,3.76
3,R0004,Aarav Nair,19,Other,Kolkata,2019-07-28,464,5,3.19
4,R0005,Diya Reddy,27,Male,Ahmedabad,2021-05-31,294,30,3.53


In [7]:
J_Data.head()

,trip_id,rider_id,zone,distance_km,duration_min,fare_amount,payment_mode,ride_date,surge_flag
0,T00001,R0037,Zone_10,11.83,74.59,104.88,Cash,2023-11-13,0
1,T00002,R0104,Zone_9,3.86,35.59,40.48,Cash,2023-07-28,1
2,T00003,R0045,Zone_8,4.70,31.03,46.39,Cash,2024-01-14,1
3,T00004,R0089,Zone_2,11.06,59.48,257.64,Cash,2023-12-13,0
4,T00005,R0003,Zone_5,7.28,67.59,72.74,UPI,2023-03-15,1


In [8]:
S_Data.head()

,zone_name,population_density,traffic_index,avg_speed_kmph,zone_type
0,Zone_1,4921,2.43,30.9,Residential
1,Zone_2,6371,0.91,58.4,Residential
2,Zone_3,12971,2.11,38.0,Business
3,Zone_4,4038,2.46,48.2,Business
4,Zone_5,2590,1.31,43.9,Business


**.info()** summary

In [9]:
C_Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   rider_id         300 non-null    object 
 1   name             300 non-null    object 
 2   age              300 non-null    int64  
 3   gender           300 non-null    object 
 4   city             300 non-null    object 
 5   signup_date      300 non-null    object 
 6   total_rides      300 non-null    int64  
 7   cancelled_rides  300 non-null    int64  
 8   avg_rating       300 non-null    float64
dtypes: float64(1), int64(3), object(5)
memory usage: 21.2+ KB


In [10]:
J_Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   trip_id       2000 non-null   object 
 1   rider_id      2000 non-null   object 
 2   zone          2000 non-null   object 
 3   distance_km   2000 non-null   float64
 4   duration_min  2000 non-null   float64
 5   fare_amount   2000 non-null   float64
 6   payment_mode  2000 non-null   object 
 7   ride_date     2000 non-null   object 
 8   surge_flag    2000 non-null   int64  
dtypes: float64(3), int64(1), object(5)
memory usage: 140.8+ KB


In [11]:
S_Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   zone_name           10 non-null     object 
 1   population_density  10 non-null     int64  
 2   traffic_index       10 non-null     float64
 3   avg_speed_kmph      10 non-null     float64
 4   zone_type           10 non-null     object 
dtypes: float64(2), int64(1), object(2)
memory usage: 532.0+ bytes


Missing value counts

In [12]:
C_Data.isna().sum()

rider_id           0
name               0
age                0
gender             0
city               0
signup_date        0
total_rides        0
cancelled_rides    0
avg_rating         0
dtype: int64

In [13]:
J_Data.isna().sum()

trip_id         0
rider_id        0
zone            0
distance_km     0
duration_min    0
fare_amount     0
payment_mode    0
ride_date       0
surge_flag      0
dtype: int64

In [14]:
S_Data.isna().sum()

zone_name             0
population_density    0
traffic_index         0
avg_speed_kmph        0
zone_type             0
dtype: int64

- Check for : 

Duplicates

In [15]:
C_Data.duplicated().sum()

np.int64(0)

In [16]:
J_Data.duplicated().sum()

np.int64(0)

In [17]:
S_Data.duplicated().sum()

np.int64(0)

Invalid enteries 

In [18]:
Under_Age = C_Data[C_Data['age'] < 20].shape[0]
Under_Age

20

In [19]:
Negative_distance = J_Data[J_Data['distance_km'] < 0].shape[0]
Negative_distance

0

In [20]:
Negative_fare = J_Data[J_Data['fare_amount'] < 0].shape[0]
Negative_fare

0

**Bonus**

In [21]:
report = ProfileReport(C_Data, title="Riders.csv")
report.to_file("Riders_EDA.html")

report = ProfileReport(J_Data, title="Trips.json")
report.to_file("Trips_EDA.html")

report = ProfileReport(S_Data, title="City_Zones.db")
report.to_file("City_Zones_EDA.html")


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 277.27it/s]


## 2. Data cleaning

- Handle numeric missing values using SimpleImputer (mean).

In [22]:
mean_imputer = SimpleImputer(strategy='mean')

Data_1_Age_mean = pd.DataFrame(
    mean_imputer.fit_transform(C_Data[['age']].values.reshape(-1, 1)),
    columns=['age']
)

In [23]:
mean_imputer = SimpleImputer(strategy='mean')

Data_1_Surge_Flag_mean = pd.DataFrame(
    mean_imputer.fit_transform(J_Data[['surge_flag']].values.reshape(-1, 1)),
    columns=['surge_flag']
)

In [24]:
mean_imputer = SimpleImputer(strategy='mean')

Data_1_PopulationDensity_mean = pd.DataFrame(
    mean_imputer.fit_transform(S_Data[['population_density']].values.reshape(-1, 1)),
    columns=['population_density']
)

- Handle categorical missing values using Most Frequent Strategy.

In [25]:
mean_imputer = SimpleImputer(strategy='most_frequent')

Data_1_rider_id_most_frequent = pd.DataFrame(
    mean_imputer.fit_transform(C_Data[['rider_id']].values.reshape(-1, 1)),
    columns=['rider_id']
)

In [26]:
mean_imputer = SimpleImputer(strategy='most_frequent')

Data_1_trip_id_most_frequent = pd.DataFrame(
    mean_imputer.fit_transform(J_Data[['trip_id']].values.reshape(-1, 1)),
    columns=['trip_id']
)

In [27]:
mean_imputer = SimpleImputer(strategy='most_frequent')

Data_1_zone_name_most_frequent = pd.DataFrame(
    mean_imputer.fit_transform(S_Data[['zone_name']].values.reshape(-1, 1)),
    columns=['zone_name']
)

- Use KNN Imputer for multivariate columns:

Trip duration , Distance , Fare Amount

In [28]:
num_cols = ['duration_min','distance_km','fare_amount']

knn_imputer = KNNImputer(
    n_neighbors=3,
    weights='distance',
)

Data3_imputed = pd.DataFrame(
    knn_imputer.fit_transform(J_Data[num_cols]),
    columns=J_Data[num_cols].columns
)

- Convert inconsistent date formats.

In [29]:
C_Data['signup_date'] = pd.to_datetime(C_Data['signup_date'], errors='coerce')
J_Data['ride_date'] = pd.to_datetime(J_Data['ride_date'], errors='coerce')

- Remove unrealistic entries:

Negative fare

In [30]:
trips = J_Data[J_Data['fare_amount'] >= 0]

Zero-distance ride but billed

In [31]:
trips = trips[~((J_Data['distance_km'] == 0) & (J_Data['fare_amount'] > 0))]

## 3. Outlier Handling

- Use Z-score method to detect fare & distance anomalies.

In [32]:
Fare_Mean = J_Data['fare_amount'].mean()
Fare_Std = J_Data['fare_amount'].std()

print("Fare Amount Mean :", Fare_Mean)
print("Fare Amount Standard Deviation :", Fare_Std)

Fare Amount Mean : 134.600375
Fare Amount Standard Deviation : 85.52198928951192


In [33]:
J_Data['Z_Score_Fare_Amount'] = (J_Data['fare_amount'] - Fare_Mean) / Fare_Std
print(J_Data[['fare_amount', 'Z_Score_Fare_Amount']].head())

   fare_amount  Z_Score_Fare_Amount
0       104.88            -0.347517
1        40.48            -1.100540
2        46.39            -1.031435
3       257.64             1.438690
4        72.74            -0.723327


In [34]:
Threshold = 3

Fare_Amount_Outliers = J_Data[(J_Data['Z_Score_Fare_Amount'].abs()) > Threshold]
print("Number of Outliers in Fare Amount:", Fare_Amount_Outliers.shape[0])

Number of Outliers in Fare Amount: 16


In [35]:
Distance_Mean = J_Data['distance_km'].mean()
Distance_Std = J_Data['distance_km'].std()

print("Distance mean : " , Distance_Mean)
print("Distance Standard Deviation : " , Distance_Std)

Distance mean :  8.24742
Distance Standard Deviation :  4.450904382090145


In [36]:
J_Data['Z_Score_Distance'] = (J_Data['distance_km'] - Distance_Mean) / Distance_Std
print(J_Data[['distance_km', 'Z_Score_Distance']].head())

   distance_km  Z_Score_Distance
0        11.83          0.804911
1         3.86         -0.985737
2         4.70         -0.797011
3        11.06          0.631912
4         7.28         -0.217354


In [37]:
Threshold = 3

Distance_Outliers = J_Data[(J_Data['Z_Score_Distance'].abs()) > Threshold]
print("Number of Outliers in Distance:", Distance_Outliers.shape[0])

Number of Outliers in Distance: 3


In [38]:
J_Data_Clean = J_Data[
    (J_Data['Z_Score_Fare_Amount'].abs() <= Threshold) &
    (J_Data['Z_Score_Distance'].abs() <= Threshold)
].copy()

In [39]:
print("Rows before outliers removing : ", len(J_Data))
print("Rows after outliers removing : ", len(J_Data_Clean))

Rows before outliers removing :  2000
Rows after outliers removing :  1981


In [40]:
print("Mean before Fare Amount Outlier Removal: ", J_Data['fare_amount'].mean())
print("Mean after Fare Amount Outlier Removal: ", J_Data_Clean['fare_amount'].mean())
print()
print("Mean before Distance Outlier Removal: ", J_Data['distance_km'].mean())
print("Mean after Distance Outlier Removal: ", J_Data_Clean['distance_km'].mean())

Mean before Fare Amount Outlier Removal:  134.600375
Mean after Fare Amount Outlier Removal:  131.93204442200908

Mean before Distance Outlier Removal:  8.24742
Mean after Distance Outlier Removal:  8.141463907117618


- Use IQR method for ride duration anomalies.

In [41]:
Q1 = J_Data['duration_min'].quantile(0.25)
Q3 = J_Data['duration_min'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
duration_outliers = J_Data[(J_Data['duration_min'] < lower) | (J_Data['duration_min'] > upper)]
print("IQR outliers (duration):", len(duration_outliers))

IQR outliers (duration): 18


In [42]:
J_Data['New_Duration'] = J_Data[J_Data['duration_min'].between(lower, upper, inclusive='both')]['duration_min']

- Apply Winsorization where necessary (e.g., extreme surge fares).

In [43]:
lower_cap = C_Data['total_rides'].quantile(0.01)
upper_cap = C_Data['total_rides'].quantile(0.99)

In [44]:
C_Data['Win_Total_Rides'] = C_Data['total_rides'].clip(lower=lower_cap, upper=upper_cap)

In [45]:
print("Mean before Total Rides Winsorization: ", C_Data['total_rides'].mean())
print("Mean after Total Rides Winsorization: ", C_Data['Win_Total_Rides'].mean())

Mean before Total Rides Winsorization:  244.69
Mean after Total Rides Winsorization:  244.6801


## 4. Data transformation

- Convert datetime → hour, day_of_week, month

In [46]:
C_Data['Year'] = C_Data['signup_date'].dt.year
C_Data['Month'] = C_Data['signup_date'].dt.month
C_Data['Day'] = C_Data['signup_date'].dt.day
C_Data['Weekday'] = C_Data['signup_date'].dt.weekday

In [47]:
C_Data[['Year','Month','Day','Weekday']]

,Year,Month,Day,Weekday
0,2020,6,29,0
1,2019,11,23,5
2,2023,5,4,3
3,2019,7,28,6
4,2021,5,31,0
...,...,...,...,...
295,2021,2,20,5
296,2023,9,20,2
297,2020,3,25,2
298,2020,9,4,4


In [48]:
J_Data['Year'] = J_Data['ride_date'].dt.year
J_Data['Month'] = J_Data['ride_date'].dt.month
J_Data['Day'] = J_Data['ride_date'].dt.day
J_Data['Weekday'] = J_Data['ride_date'].dt.weekday

In [49]:
J_Data[['Year','Month','Day','Weekday']]

,Year,Month,Day,Weekday
0,2023,11,13,0
1,2023,7,28,4
2,2024,1,14,6
3,2023,12,13,2
4,2023,3,15,2
...,...,...,...,...
1995,2023,3,20,0
1996,2024,6,25,1
1997,2024,3,23,5
1998,2023,10,29,6


- Encode categorical columns:

Label Encode: gender

In [50]:
Le_gender = LabelEncoder()
C_Data['gender_encoded'] = Le_gender.fit_transform(C_Data['gender'])

One-Hot Encode: ride_payment_mode, zone_name

In [51]:
ohe_payment = OneHotEncoder(
    drop='first',            
    sparse_output=False,     
    handle_unknown='ignore'  
)


In [52]:
Ride_Payment_Mode = ohe_payment.fit_transform(J_Data[['payment_mode']])
payment_cols = ohe_payment.get_feature_names_out(['payment_mode'])
payment_df = pd.DataFrame(Ride_Payment_Mode, columns=payment_cols, index=J_Data.index)

In [53]:
J_Data = pd.concat([J_Data, payment_df], axis=1)

In [54]:
ohe_zones = OneHotEncoder(
    drop='first',            
    sparse_output=False,     
    handle_unknown='ignore'  
)

In [55]:
ohe_zone = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
Zone_Name = ohe_zone.fit_transform(S_Data[['zone_name']])

In [56]:
zone_cols = ohe_zone.get_feature_names_out(['zone_name'])
zone_df = pd.DataFrame(Zone_Name, columns=zone_cols, index=S_Data.index)

In [57]:
S_Data = pd.concat([S_Data, zone_df], axis=1)

Ordinal Encode: traffic_level (Low < Medium < High)

In [58]:
S_Data.head(2)

,zone_name,population_density,traffic_index,avg_speed_kmph,zone_type,zone_name_Zone_10,zone_name_Zone_2,zone_name_Zone_3,zone_name_Zone_4,zone_name_Zone_5,zone_name_Zone_6,zone_name_Zone_7,zone_name_Zone_8,zone_name_Zone_9
0,Zone_1,4921,2.43,30.9,Residential,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Zone_2,6371,0.91,58.4,Residential,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [59]:
S_Data['traffic_level'] = pd.qcut(S_Data['traffic_index'], q=3, labels=['Low', 'Medium', 'High'])

enc = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
S_Data['traffic_level_encoded'] = enc.fit_transform(S_Data[['traffic_level']])

S_Data[['traffic_index', 'traffic_level', 'traffic_level_encoded']]

,traffic_index,traffic_level,traffic_level_encoded
0,2.43,High,2.0
1,0.91,Low,0.0
2,2.11,High,2.0
3,2.46,High,2.0
4,1.31,Medium,1.0
5,0.54,Low,0.0
6,0.64,Low,0.0
7,1.93,High,2.0
8,1.83,Medium,1.0
9,1.54,Medium,1.0


- Binning

Customer ride frequency (Low/Med/High)

In [60]:
C_Data['ride_frequency_group'] = pd.cut(
    C_Data['total_rides'],
    bins=[0, 5, 15, float('inf')],
    labels=['Low', 'Medium', 'High'],
    right=True
)

print("Bin Distribution:")
print(C_Data['ride_frequency_group'].value_counts())
print()
print("Ride Frequency Binning (Custom Edges):")
C_Data[['rider_id', 'total_rides', 'ride_frequency_group']]

Bin Distribution:
ride_frequency_group
High      291
Medium      8
Low         1
Name: count, dtype: int64

Ride Frequency Binning (Custom Edges):


,rider_id,total_rides,ride_frequency_group
0,R0001,56,High
1,R0002,70,High
2,R0003,45,High
3,R0004,464,High
4,R0005,294,High
...,...,...,...
295,R0296,390,High
296,R0297,396,High
297,R0298,65,High
298,R0299,457,High


- Transform skewed numeric columns:

Log transform on fare and distance

In [61]:
print("SKEWNESS COMPARISON - FARE AMOUNT")
print(f"Original skewness : {J_Data['fare_amount'].skew():.3f}")

log_transformer_fare = FunctionTransformer(
    func=np.log1p,
    inverse_func=np.expm1,
    validate=True
)

J_Data['fare_amount_log'] = log_transformer_fare.fit_transform(
    J_Data[['fare_amount']]
)

print(f"Log transformed skewness : {J_Data['fare_amount_log'].skew():.3f}")
print()
J_Data[['fare_amount', 'fare_amount_log']]

SKEWNESS COMPARISON - FARE AMOUNT
Original skewness : 0.774
Log transformed skewness : -1.515



c:\PJ Things\AI ML and Data Science\Data Pre-processing\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but FunctionTransformer was fitted with feature names
  warnings.warn(


,fare_amount,fare_amount_log
0,104.88,4.662306
1,40.48,3.725211
2,46.39,3.858411
3,257.64,5.555437
4,72.74,4.300545
...,...,...
1995,22.12,3.140698
1996,193.75,5.271717
1997,58.45,4.085136
1998,74.92,4.329680


In [62]:
print("SKEWNESS COMPARISON - DISTANCE")
print(f"Original skewness : {J_Data['distance_km'].skew():.3f}")

log_transformer_distance = FunctionTransformer(
    func=np.log1p,
    inverse_func=np.expm1,
    validate=True
)

J_Data['distance_km_log'] = log_transformer_distance.fit_transform(
    J_Data[['distance_km']]
)

print(f"Log transformed skewness : {J_Data['distance_km_log'].skew():.3f}")
print()
J_Data[['distance_km', 'distance_km_log']]

SKEWNESS COMPARISON - DISTANCE
Original skewness : 0.231
Log transformed skewness : -1.090



c:\PJ Things\AI ML and Data Science\Data Pre-processing\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but FunctionTransformer was fitted with feature names
  warnings.warn(


,distance_km,distance_km_log
0,11.83,2.551786
1,3.86,1.581038
2,4.70,1.740466
3,11.06,2.489894
4,7.28,2.113843
...,...,...
1995,0.91,0.647103
1996,11.40,2.517696
1997,4.94,1.781709
1998,7.76,2.170196


Square-root transform on duration

In [63]:
sqrt_transformer = FunctionTransformer(
    func=np.sqrt,
    inverse_func=np.square
)

J_Data['duration_min_sqrt'] = sqrt_transformer.transform(J_Data[['duration_min']])

J_Data['duration_min_restored'] = sqrt_transformer.inverse_transform(
    J_Data[['duration_min_sqrt']]
).round(0)

J_Data[['trip_id', 'duration_min', 'duration_min_sqrt', 'duration_min_restored']]

,trip_id,duration_min,duration_min_sqrt,duration_min_restored
0,T00001,74.59,8.636550,75.0
1,T00002,35.59,5.965735,36.0
2,T00003,31.03,5.570458,31.0
3,T00004,59.48,7.712328,59.0
4,T00005,67.59,8.221314,68.0
...,...,...,...,...
1995,T01996,6.44,2.537716,6.0
1996,T01997,103.03,10.150369,103.0
1997,T01998,39.32,6.270566,39.0
1998,T01999,51.43,7.171471,51.0


## 5. Feature calculation

- Scale numeric fetures using : 

In [64]:
scale_cols = ['fare_amount', 'distance_km', 'duration_min']

print("Before Scaling : ")
print(J_Data[scale_cols].agg(['mean', 'std', 'min', 'max']))

Before Scaling : 
      fare_amount  distance_km  duration_min
mean   134.600375     8.247420     61.883410
std     85.521989     4.450904     35.980603
min      0.250000     0.020000      0.190000
max    472.290000    25.860000    201.070000


Standard scaler

In [65]:
standard_scaler = StandardScaler()
J_Data_standard = pd.DataFrame(standard_scaler.fit_transform(J_Data[scale_cols]) , columns=[c + '_standard' for c in scale_cols], index=J_Data.index)

In [66]:
print ("After Standard Scaling : ")
print(J_Data_standard.agg(['mean', 'std', 'min', 'max']))

After Standard Scaling : 
      fare_amount_standard  distance_km_standard  duration_min_standard
mean         -1.616485e-16          5.329071e-18          -1.172396e-16
std           1.000250e+00          1.000250e+00           1.000250e+00
min          -1.571338e+00         -1.848945e+00          -1.715058e+00
max           3.949558e+00          3.958069e+00           3.869346e+00


MInMax Scaler 

In [67]:
minmax_scaler = MinMaxScaler()
J_Data_minmax = pd.DataFrame(minmax_scaler.fit_transform(J_Data[scale_cols]) , columns=[c + '_minmax' for c in scale_cols], index=J_Data.index)

In [68]:
print("After Standard Scaling : ")
print(J_Data_minmax.agg(['mean', 'std', 'min', 'max']))

After Standard Scaling : 
      fare_amount_minmax  distance_km_minmax  duration_min_minmax
mean            0.284617            0.318399             0.307116
std             0.181175            0.172249             0.179115
min             0.000000            0.000000             0.000000
max             1.000000            1.000000             1.000000


In [69]:
J_Data = pd.concat([J_Data, J_Data_standard, J_Data_minmax], axis=1)

## 6. Feature construction

Average Ride Distance & Average Ride Fare

In [70]:
C_Data_agg = J_Data.groupby('rider_id').agg(
    total_distance=('distance_km', 'sum'),
    total_fare=('fare_amount', 'sum'),
    trip_count=('trip_id', 'count')
).reset_index()

In [71]:
C_Data_agg['avg_ride_distance'] = C_Data_agg['total_distance'] / C_Data_agg['trip_count']
C_Data_agg['avg_ride_fare'] = C_Data_agg['total_fare'] / C_Data_agg['trip_count']

C_Data = C_Data.merge(C_Data_agg[['rider_id', 'avg_ride_distance', 'avg_ride_fare']], on='rider_id', how='left')

In [87]:
print(C_Data[['rider_id', 'avg_ride_distance', 'avg_ride_fare']])

    rider_id  avg_ride_distance  avg_ride_fare
0      R0001          10.420000     158.785000
1      R0002          10.812857     167.944286
2      R0003           6.768000      97.171000
3      R0004           6.201667     113.838333
4      R0005           7.322857     132.350000
..       ...                ...            ...
295    R0296          11.351429     187.832857
296    R0297           5.930000     100.645000
297    R0298           9.027143     143.201429
298    R0299           7.472500     115.300000
299    R0300           9.146667     166.128333

[300 rows x 3 columns]


Is Peak Hour

In [75]:
J_Data['is_peak_hour'] = J_Data['ride_date'].dt.hour.between(7, 9) | J_Data['ride_date'].dt.hour.between(17, 19)

In [88]:
print(J_Data['is_peak_hour'])

0       False
1       False
2       False
3       False
4       False
        ...  
1995    False
1996    False
1997    False
1998    False
1999    False
Name: is_peak_hour, Length: 2000, dtype: bool


Days Since Signup

In [79]:
C_Data['days_since_signup'] = (pd.Timestamp.now() - C_Data['signup_date']).dt.days

In [90]:
print(C_Data['days_since_signup'])

0      2268
1      2487
2      1229
3      2605
4      1932
       ... 
295    2032
296    1090
297    2364
298    2201
299    1164
Name: days_since_signup, Length: 300, dtype: int64


Ride Cancellation Rate

In [81]:
C_Data['ride_cancellation_rate'] = C_Data['cancelled_rides'] / C_Data['total_rides']

In [91]:
print(C_Data['ride_cancellation_rate'])

0      0.000000
1      0.071429
2      0.200000
3      0.010776
4      0.102041
         ...   
295    0.148718
296    0.116162
297    0.169231
298    0.000000
299    0.024055
Name: ride_cancellation_rate, Length: 300, dtype: float64


New Surge Flag

In [84]:
threshold = (J_Data['fare_amount'] / J_Data['distance_km'].quantile(0.75))
J_Data['surge_flag_check'] = ((J_Data['fare_amount'] / J_Data['distance_km']) > threshold).astype(int)

In [92]:
print(J_Data['surge_flag_check'])

0       0
1       1
2       1
3       1
4       1
       ..
1995    1
1996    0
1997    1
1998    1
1999    0
Name: surge_flag_check, Length: 2000, dtype: int64


## 7. Final dataset - Merge & Export

- Merged C_data , J_data & S_Data

In [93]:
Final_Data = J_Data.merge(C_Data, on='rider_id', how='left')
Final_Data = Final_Data.merge(S_Data, left_on='zone', right_on='zone_name', how='left')

print("Final dataset shape:", Final_Data.shape)
Final_Data.head()

Final dataset shape: (2000, 66)


,trip_id,rider_id,zone,distance_km,duration_min,fare_amount,payment_mode,ride_date,surge_flag,Z_Score_Fare_Amount,...,zone_name_Zone_2,zone_name_Zone_3,zone_name_Zone_4,zone_name_Zone_5,zone_name_Zone_6,zone_name_Zone_7,zone_name_Zone_8,zone_name_Zone_9,traffic_level,traffic_level_encoded
0,T00001,R0037,Zone_10,11.83,74.59,104.88,Cash,2023-11-13,0,-0.347517,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Medium,1.0
1,T00002,R0104,Zone_9,3.86,35.59,40.48,Cash,2023-07-28,1,-1.100540,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Medium,1.0
2,T00003,R0045,Zone_8,4.70,31.03,46.39,Cash,2024-01-14,1,-1.031435,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,High,2.0
3,T00004,R0089,Zone_2,11.06,59.48,257.64,Cash,2023-12-13,0,1.438690,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Low,0.0
4,T00005,R0003,Zone_5,7.28,67.59,72.74,UPI,2023-03-15,1,-0.723327,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,Medium,1.0


- Summary Table

In [94]:
Rows_Before = len(J_Data)
Rows_After = len(Final_Data)

Missing_Before = C_Data.isna().sum().sum() + J_Data.isna().sum().sum() + S_Data.isna().sum().sum()
Missing_After = Final_Data.isna().sum().sum()

Outliers_Removed = len(J_Data) - len(J_Data_Clean)

New_Features = ['avg_ride_distance', 'avg_ride_fare', 'is_peak_hour', 'days_since_signup',
                'ride_cancellation_rate', 'surge_flag_check']

Summary = pd.DataFrame({
    'Metric': ['Rows', 'Missing Values', 'Outliers Removed', 'New Engineered Features'],
    'Before': [Rows_Before, Missing_Before, 0, 0],
    'After': [Rows_After, Missing_After, Outliers_Removed, len(New_Features)]
})

print(Summary)

                    Metric  Before  After
0                     Rows    2000   2000
1           Missing Values      20     18
2         Outliers Removed       0     19
3  New Engineered Features       0      6


- Final prepared rides Dataset

In [95]:
Final_Data.to_csv('final_prepared_rides_dataset.csv', index=False)
print("Saved final_prepared_rides_dataset.csv")

Saved final_prepared_rides_dataset.csv


## 8. Visualizations

### Surge vs No-Surge trip patterns

In [105]:
plt.figure(figsize=(6, 4))
J_Data.groupby('surge_flag')['fare_amount'].mean().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Avg Fare: Surge vs No-Surge')
plt.xlabel('Surge Flag')
plt.ylabel('Average Fare')
plt.tight_layout()
plt.savefig('surge_vs_nosurge.png')
plt.show()

C:\Users\bhatt\AppData\Local\Temp\ipykernel_8888\2078017074.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
